# EduPredict AI — Exploratory Data Analysis & Model Training

This notebook walks through the full machine learning pipeline for predicting
student grades: cleaning, feature engineering, encoding, scaling, training
three models, and evaluating them.

Every important concept is explained briefly as we go — this notebook is
meant to be read and understood, not just run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath(".."))

from utils import (
    load_data, clean_data, engineer_features, encode_features, scale_features,
    CATEGORICAL_COLS, NUMERIC_COLS, TARGET_COL
)

pd.set_option("display.max_columns", None)
os.chdir("..")  # so relative paths (data/, plots/) resolve from project root

## 1. Load the raw dataset

We use the Student Performance dataset (UCI/Kaggle-style schema). This
sandbox environment had no internet access, so `generate_dataset.py` was
used to synthesize a dataset with the same columns, realistic value ranges,
and a real underlying signal (better attendance/study hours/fewer failures
→ better grades, plus noise) — to swap in the real dataset, just replace
`data/student.csv` with matching column names.

In [ ]:
df = load_data()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Data Cleaning

**Duplicates** are removed because repeated rows would let the model see
the same example multiple times, artificially inflating its confidence on
that pattern.

**Missing values** are filled rather than dropped, so we don't throw away
otherwise-useful rows:
- Numeric columns → filled with the **median** (robust to outliers, unlike
  the mean).
- Categorical columns → filled with the **mode** (most frequent category).

We also drop `final_score`, since it's the exact value the synthetic grade
was derived from — keeping it would let the model "cheat" instead of
learning genuine patterns (this is called **data leakage**).

In [ ]:
df_clean = clean_data(df)
print("Before:", df.shape, " After:", df_clean.shape)
df_clean.isnull().sum()

## 3. Feature Engineering

We add two simple, explainable features:
- **study_efficiency** = weekly_study_hours × (attendance_percentage / 100)
  — captures that study time only helps if the student is actually present.
- **academic_risk** = 1 if failures ≥ 2, else 0 — a simple flag for
  at-risk students.

We deliberately keep feature engineering simple: a couple of well-reasoned
features beat many arbitrary ones, and simple features are far easier to
explain in an interview.

In [ ]:
df_feat = engineer_features(df_clean)
df_feat[["weekly_study_hours", "attendance_percentage", "study_efficiency", "failures", "academic_risk"]].head()

## 4. Visualizations

All plots use **Matplotlib only** (no Seaborn/Plotly), per project
constraints.

In [ ]:
plt.figure(figsize=(6,4))
df_feat[TARGET_COL].value_counts().sort_index().plot(kind="bar", color="#4C72B0")
plt.title("Grade Distribution")
plt.xlabel("Grade"); plt.ylabel("Number of Students")
plt.tight_layout(); plt.show()

Grades are **imbalanced** — far more C/D students than A students. This is
realistic (most real grade distributions skew this way) and is an important
reason we use **macro-averaged F1 score** later instead of plain accuracy:
accuracy alone could look good just by always predicting the majority class.

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df_feat["attendance_percentage"], bins=20, color="#55A868", edgecolor="black")
plt.title("Attendance Distribution")
plt.xlabel("Attendance (%)"); plt.ylabel("Number of Students")
plt.tight_layout(); plt.show()

In [ ]:
numeric_df = df_feat[NUMERIC_COLS + ["study_efficiency", "academic_risk"]]
corr = numeric_df.corr()

plt.figure(figsize=(7,6))
im = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.show()

## 5. Encoding Categorical Features

**Label Encoding** converts each category into an integer (e.g. "Yes" → 1,
"No" → 0). We use it here because all our categorical columns are binary
or have a natural order (parent education: High School < Bachelors <
Masters < PhD), so a single integer column is a clean, simple
representation.

**One-Hot Encoding** is the alternative — it creates a separate 0/1 column
per category instead. It's better when categories have **no natural
order** and there are few of them (e.g. "gender" could arguably use it
too). Below we demonstrate it for comparison, purely for learning purposes
— the actual pipeline in `utils.py` uses Label Encoding for simplicity and
consistency with the tree-based models.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

demo_ohe = OneHotEncoder(sparse_output=False)
demo_encoded = demo_ohe.fit_transform(df_feat[["gender"]])
pd.DataFrame(demo_encoded, columns=demo_ohe.get_feature_names_out(["gender"])).head()

In [ ]:
df_encoded, encoders = encode_features(df_feat, fit=True)
df_encoded[CATEGORICAL_COLS].head()

## 6. Scaling Numeric Features

**StandardScaler** transforms each numeric column to have mean 0 and
standard deviation 1: `z = (x - mean) / std`.

**Why some models need it:** Logistic Regression relies on gradient-based
optimization over a weighted sum of features. A feature like
`attendance_percentage` (range 0-100) would dominate a feature like
`failures` (range 0-3) unless both are put on the same scale.

**Why Random Forest / Decision Tree don't need it:** these models split
data using single-feature threshold rules (e.g. "attendance_percentage >
70?"). The relative order of values matters, not their scale — so scaling
changes nothing about which splits get chosen.

In [ ]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
df_encoded[TARGET_COL] = target_encoder.fit_transform(df_encoded[TARGET_COL])

feature_cols = NUMERIC_COLS + CATEGORICAL_COLS + ["study_efficiency", "academic_risk"]
X = df_encoded[feature_cols]
y = df_encoded[TARGET_COL]

X_scaled, scaler = scale_features(X, NUMERIC_COLS + ["study_efficiency"], fit=True)
X_scaled.describe().loc[["mean", "std"]]

## 7. Train / Test Split

We use an 80/20 split with **stratification** on the target, so the class
proportions (A/B/C/D/F) are preserved in both the train and test sets —
important given the class imbalance we saw earlier.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 8. Train Three Models

- **Logistic Regression** — a linear baseline. Fast, interpretable
  (`class_weight="balanced"` helps counter the class imbalance).
- **Decision Tree** — learns simple if/else rules; easy to visualize.
- **Random Forest** — an ensemble of many decision trees; usually the most
  accurate because averaging reduces overfitting compared to one tree.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
print("Trained:", list(models.keys()))

## 9. Evaluation Metrics

- **Accuracy** — % of predictions that were exactly correct. Can be
  misleading with imbalanced classes.
- **Precision** — of everyone predicted as grade X, how many really were X.
  High precision = few false alarms.
- **Recall** — of everyone who really was grade X, how many the model
  found. High recall = few missed cases.
- **F1 Score** — the harmonic mean of precision and recall; a single
  balanced number, especially useful here given the class imbalance.
- **Confusion Matrix** — a table showing exactly which grades get confused
  with which other grades.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, average="macro", zero_division=0),
        "recall": recall_score(y_test, preds, average="macro", zero_division=0),
        "f1": f1_score(y_test, preds, average="macro", zero_division=0),
    }

pd.DataFrame(results).T.round(3)

In [ ]:
best_name = max(results, key=lambda n: results[n]["f1"])
best_model = models[best_name]
print("Best model by macro F1:", best_name)

cm = confusion_matrix(y_test, best_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
fig, ax = plt.subplots(figsize=(6,5))
disp.plot(ax=ax, cmap="Blues", colorbar=True)
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout(); plt.show()

## 10. Conclusion

The best-performing model (by macro F1 score) is saved along with its
preprocessing artifacts (encoders + scaler) via `train_model.py`, so the
Streamlit app can load it directly for live predictions — see
`models/best_model.pkl`.

**Key takeaways for an interview:**
- Random Forest and Logistic Regression are compared fairly using
  **macro F1**, not just accuracy, because of class imbalance.
- All preprocessing (`clean_data`, `engineer_features`, `encode_features`,
  `scale_features`) lives in `utils.py` and is reused identically at
  training time and prediction time — avoiding train/serve skew.
- Feature importance and confusion matrix plots make the model's behavior
  explainable, not just accurate.